In [1]:
using Oceananigans
using Oceananigans.Units
using Printf

In [2]:
Lx = 1000kilometers # east-west extent [m]
Ly = 1000kilometers # north-south extent [m]
Lz = 1kilometers    # depth [m]

# A mountain in the ocean
mountain_h₀ = 250meters
mountain_width = 500kilometers
#bottom(x, y) = Lz * ( 0.8 - exp(-(x^2 + y^2) / 2mountain_width^2) )
bathtub_shape(x, y) = exp(-(x^2 + y^2) / 2mountain_width^2)

bottom(x, y) = -Lz + 0 * Lz * bathtub_shape(x, y)

grid = RectilinearGrid(size = (48, 48, 8),
                       x = (-Lx/2, Lx/2),
                       y = (-Ly/2, Ly/2),
                       z = (-Lz, 0),
                       topology = (Bounded, Bounded, Bounded),
                       halo = (4, 4, 4),
)

grid = ImmersedBoundaryGrid(grid, PartialCellBottom(bottom))

48×48×8 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CPU with 4×4×4 halo:
├── immersed_boundary: PartialCellBottom(mean(zb)=-795.918, min(zb)=-1000.0, max(zb)=0.0, ϵ=0.2)
├── underlying_grid: 48×48×8 RectilinearGrid{Float64, Bounded, Bounded, Bounded} on CPU with 4×4×4 halo
├── Bounded  x ∈ [-500000.0, 500000.0] regularly spaced with Δx=20833.3
├── Bounded  y ∈ [-500000.0, 500000.0] regularly spaced with Δy=20833.3
└── Bounded  z ∈ [-1000.0, 0.0]        regularly spaced with Δz=125.0

In [3]:
x = xnodes(grid, Center())
y = ynodes(grid, Center())

bottom_boundary = interior(grid.immersed_boundary.bottom_height)[:, :, 1]
top_boundary = 0 * x

print("size(bottom_boundary) = ", size(bottom_boundary))

using CairoMakie
fig = Figure(size = (700, 700))

a1 = Axis3(fig[1, 1], title = "Topography")
contour3d!(a1, x, y, bottom_boundary, levels = 10)

#surface!(a1, x, y, bottom_boundary,
#    colormap = :darkterrain,
#    colorrange = (80, 190),
#)


fig




size(bottom_boundary) = (48, 48)

LoadError: Failed to resolve [31m_positions:[39m
[ComputeEdge] [31m_positions, input_text[39m = #register_arguments!##0(([92mposition[39m, [31mtext[39m, [90marg1[39m, ), changed, cached)
[90m  @ /home/xtt/.julia/packages/Makie/Vn16E/src/basic_recipes/text.jl:54[39m
[ComputeEdge] [31mtext[39m = (::InputFunctionWrapper(:text, AttributeConvert{text, text}))(([31mtext_strings[39m, ), changed, cached)
[90m  @ /home/xtt/.julia/packages/Makie/Vn16E/src/conversions.jl:920[39m
[ComputeEdge] [31mtext_strings[39m = (::MapFunctionWrapper(#plot!##58))(([31mcomputed_levels[39m, [90mlabelformatter[39m, ), changed, cached)
[90m  @ unknown method location[39m
[ComputeEdge] [31mcontour_points, contour_colors, computed_levels, lbl_pos1, lbl_pos2, lbl_pos3, computed_lbl_colors[39m = (::MapFunctionWrapper(#plot!##52))(([92mconverted_1[39m, [92mconverted_2[39m, [92mconverted_3[39m, [92mzlevels[39m, [31mlevel_colors[39m, [90mlabels[39m, ), changed, cached)
[90m  @ /ho

## Construct F-plane and tidal forcing

In [4]:
coriolis = FPlane(latitude = -45)

FPlane{Float64}(f=-0.000103126)

In [5]:
T₂ = 12.421hours
ω₂ = 2π / T₂ # radians/sec
ϵ = 0.1 # excursion parameter
U₂ = ϵ * ω₂ * Lx
A₂ = U₂ * (ω₂^2 - coriolis.f^2) / ω₂

@inline tidal_forcing(x, y, z, t, p) = p.A₂ * sin(p.ω₂ * t)
u_forcing = Forcing(tidal_forcing, parameters=(; A₂, ω₂))

ContinuousForcing{@NamedTuple{A₂::Float64, ω₂::Float64}}
├── func: tidal_forcing (generic function with 1 method)
├── parameters: (A₂ = 0.0009109305874754121, ω₂ = 0.00014051439111137024)
└── field dependencies: ()

In [6]:
model = HydrostaticFreeSurfaceModel(; grid, coriolis,
                                      buoyancy = BuoyancyTracer(),
                                      tracers = :b,
                                      momentum_advection = WENO(),
                                      tracer_advection = WENO(),
                                      forcing = (; u = u_forcing))

HydrostaticFreeSurfaceModel{CPU, ImmersedBoundaryGrid}(time = 0 seconds, iteration = 0)
├── grid: 48×48×8 ImmersedBoundaryGrid{Float64, Bounded, Bounded, Bounded} on CPU with 4×4×4 halo
├── timestepper: QuasiAdamsBashforth2TimeStepper
├── tracers: b
├── closure: Nothing
├── buoyancy: BuoyancyTracer with ĝ = NegativeZDirection()
├── free surface: SplitExplicitFreeSurface with gravitational acceleration 9.80665 m s⁻²
│   └── substepping: FixedTimeStepSize(1.736 minutes)
├── advection scheme: 
│   ├── momentum: WENO{3, Float64, Float32}(order=5)
│   └── b: WENO{3, Float64, Float32}(order=5)
├── vertical_coordinate: ZCoordinate
└── coriolis: FPlane{Float64}

In [7]:
Nᵢ² = 1e-4  # [s⁻²] initial buoyancy frequency / stratification
bᵢ(x, y, z) = Nᵢ² * z
set!(model, u=U₂, b=bᵢ)

In [8]:
Δt = 5minutes
stop_time = 4days
simulation = Simulation(model; Δt, stop_time)

using Printf
wall_clock = Ref(time_ns())
function progress(sim)
    elapsed = 1e-9 * (time_ns() - wall_clock[])
    msg = @sprintf("Iter: %d, time: %s, wall time: %s, max|w|: %6.3e, m s⁻¹\n",
                   iteration(sim), prettytime(sim), prettytime(elapsed),
                   maximum(abs, sim.model.velocities.w))
    wall_clock[] = time_ns()
    @info msg
    return nothing
end
add_callback!(simulation, progress, name=:progress, IterationInterval(200))



In [9]:
b = model.tracers.b
u, v, w = model.velocities
U = Field(Average(u))
u′ = u - U
N² = ∂z(b)

filename = "internal_tide"
save_fields_interval = 30minutes

simulation.output_writers[:fields] = JLD2Writer(model, (; u, u′, w, b, N²); filename,
                                                schedule = TimeInterval(save_fields_interval),
                                                overwrite_existing = true)

JLD2Writer scheduled on TimeInterval(30 minutes):
├── filepath: internal_tide.jld2
├── 5 outputs: (u, u′, w, b, N²)
├── array_type: Array{Float32}
├── including: [:grid, :coriolis, :buoyancy, :closure]
├── file_splitting: NoFileSplitting
└── file size: 115.0 KiB

In [10]:
run!(simulation)

[ Info: Initializing simulation...
[ Info: Iter: 0, time: 0 seconds, wall time: 45.857 seconds, max|w|: 6.745e-01, m s⁻¹
[ Info:     ... simulation initialization complete (15.182 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (3.434 seconds).
[ Info: Iter: 200, time: 16.667 hours, wall time: 7.599 seconds, max|w|: 7.714e-02, m s⁻¹
[ Info: Iter: 400, time: 1.389 days, wall time: 2.290 seconds, max|w|: 4.372e-02, m s⁻¹
[ Info: Iter: 600, time: 2.083 days, wall time: 2.283 seconds, max|w|: 8.186e-02, m s⁻¹
[ Info: Iter: 800, time: 2.778 days, wall time: 2.584 seconds, max|w|: 4.228e-02, m s⁻¹
[ Info: Iter: 1000, time: 3.472 days, wall time: 2.472 seconds, max|w|: 5.540e-02, m s⁻¹
[ Info: Simulation is stopping after running for 33.485 seconds.
[ Info: Simulation time 4 days equals or exceeds stop time 4 days.


## Loading Output

In [11]:
saved_output_filename = filename * ".jld2"

u′_t = FieldTimeSeries(saved_output_filename, "u′")
 w_t = FieldTimeSeries(saved_output_filename, "w")
N²_t = FieldTimeSeries(saved_output_filename, "N²")

umax = maximum(abs, u′_t[end])
wmax = maximum(abs, w_t[end])

times = u′_t.times

0.0:1800.0:345600.0

In [39]:
using CairoMakie


n = Observable(1)

title = @lift @sprintf("t = %1.2f days = %1.2f T₂",
                       round(times[$n] / day, digits=2) , round(times[$n] / T₂, digits=2))

u′ₙ = @lift u′_t[$n]
 wₙ = @lift  w_t[$n]
N²ₙ = @lift N²_t[$n]

axis_kwargs = (xlabel = "x [m]",
               ylabel = "z [m]",
               limits = ((-grid.Lx/2, grid.Lx/2), (-grid.Lz, 0)),
               titlesize = 20)

print("Here")
fig = Figure(size = (700, 900))

fig[1, :] = Label(fig, title, fontsize=24, tellwidth=false)
print("Here2")

ax_u = Axis(fig[2, 1]; title = "u'-velocity", axis_kwargs...)
print(size(u′ₙ[][:, 24, :]))
print(typeof(u′ₙ[][:, 24, :]))

hm_u = heatmap!(ax_u, u′ₙ[][:, 24, :]; nan_color=:gray, colorrange=(-umax, umax), colormap=:balance)
#Colorbar(fig[2, 2], hm_u, label = "m s⁻¹")
#=
print("Here3")
ax_w = Axis(fig[3, 1]; title = "w-velocity", axis_kwargs...)
hm_w = heatmap!(ax_w, wₙ; nan_color=:gray, colorrange=(-wmax, wmax), colormap=:balance)
Colorbar(fig[3, 2], hm_w, label = "m s⁻¹")
print("Here4")
ax_N² = Axis(fig[4, 1]; title = "stratification N²", axis_kwargs...)
hm_N² = heatmap!(ax_N², N²ₙ; nan_color=:gray, colorrange=(0.9Nᵢ², 1.1Nᵢ²), colormap=:magma)
Colorbar(fig[4, 2], hm_N², label = "s⁻²")
print("Going to call")
=#
fig

HereHere2(57, 16)OffsetArrays.OffsetMatrix{Float64, Matrix{Float64}}

DimensionMismatch: DimensionMismatch: axes must agree, got (Base.OneTo(16), Base.OneTo(57)) and (OffsetArrays.IdOffsetRange(values=-3:12, indices=-3:12), OffsetArrays.IdOffsetRange(values=-3:53, indices=-3:53))

In [25]:
n=2


2